## Libraries

In [0]:
%pip install openpyxl adjustText

In [0]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import openpyxl
import math
import mlflow
from matplotlib.ticker import FuncFormatter, LogLocator, NullFormatter
import matplotlib as mpl
from adjustText import adjust_text
from matplotlib.dates import date2num
import matplotlib.dates as mdates
from matplotlib.dates import date2num, WeekdayLocator, DateFormatter, MO

mlflow.autolog(disable=True)

## Data

### Reading

In [0]:
directory = '/Volumes/cpx_ml_silver_dev/marketing_mix_modelling/data_v1'

# Website visits (Cineplex, Rec Room, Playdium)
website = pd.read_csv(f"{directory}/clean_data/website.csv")
# Cineclub signups and exits
cineclub = pd.read_csv(f"{directory}/clean_data/cineclub.csv")
# Attendance and revenue
attendance_and_revenue = pd.read_csv(f"{directory}/clean_data/attendance_and_revenue.csv")
# SEM spend
sem = pd.read_csv(f"{directory}/clean_data/sem.csv")
sem_new = pd.read_excel(f'{directory}/raw_data/SEM EXH & LBE Mar 2023 to Jun 2025 weekly report_v2.xlsx', skiprows=2)
# Meta spend
meta = pd.read_csv(f"{directory}/clean_data/meta.csv")
# Datorama programmatic
programmatic = pd.read_csv(f"{directory}/clean_data/programmatic.csv")
# Datorama social
social = pd.read_csv(f"{directory}/clean_data/social.csv")
# Email campaigns
email = pd.read_csv(f"{directory}/clean_data/email.csv")

### Cleaning

In [0]:
for i in [website, cineclub, attendance_and_revenue, sem, sem_new, meta, programmatic, social, email]:
    i.columns = i.columns.str.lower().str.replace(' ', '_')
    i['week'] = pd.to_datetime(i['week'])

In [0]:
start = '2023-03-31'
end = '2025-03-31'

website = website[website['week'].between(start, end)]
cineclub = cineclub[cineclub['week'].between(start, end)]
attendance_and_revenue = attendance_and_revenue[attendance_and_revenue['week'].between(start, end)]
sem = sem[sem['week'].between(start, end)]
meta = meta[meta['week'].between(start, end)]
programmatic = programmatic[programmatic['week'].between(start, end)]
social = social[social['week'].between(start, end)]

In [0]:
col = meta['result_indicator']
conditions = [col.str.contains('reach', na=False), 
              col.str.contains('link_click|landing_page_view', na=False), 
              col.str.contains('offsite_conversion|subscribe_website', na=False)]
values = ['Awareness', 'Consideration', 'Conversion']

meta['objective'] = np.select(conditions, values, default=np.NaN)
meta['lob'] = meta['campaign_name'].str.extract(r'(?:CEC|CEA)_(.*?)_')

meta = meta[meta['type']!='LBE']

## EDA

In [0]:
palette = ['#065CB1', '#E5C224', 'grey', '#CA0707', '#1F1F1F']

### SEM Spend

#### Awareness Only

In [0]:
sem_awa = sem[sem['campaign'].str.contains('_AW|Awareness')].copy()

**NOTE:** No data for awareness campaigns in analysis period (Q2 2023 to Q1 2025)

#### Consideration Only

In [0]:
sem_cons = sem[sem['campaign'].str.contains('_CONS_|Consideration')].copy()

In [0]:
# Data Preparation
tmp = sem_cons.copy()
tmp = tmp[~tmp['lob'].isin(['TRR', 'PDM'])]
tmp['lob'] = np.where(tmp['campaign'].str.contains('Cineplex Pictures'), 'Film', tmp['lob'])
tmp['lob'] = np.where(tmp['lob'].isin(['Exhibition ', 'Film', 'G&E']), tmp['lob'], 'Prepaid')
tmp = tmp.groupby('lob').agg({'cost': 'sum', 'clicks': 'sum'}).sort_values(by='cost', ascending=False)
tmp = tmp / tmp.sum() * 100

# Visualization
indices = np.arange(len(tmp))
fig = plt.figure(figsize=(8, 6), dpi=240)
bars1 = plt.bar(indices - 0.15, tmp['cost'], width=0.3, color=palette[0], zorder=2, label='Spend (% of Total)')
bars2 = plt.bar(indices + 0.15, tmp['clicks'], width=0.3, color=palette[1], zorder=2, label='Clicks (% of Total)')

for bar in bars1 + bars2:
    yval = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2.0, yval + 1, f'{yval:.0f}%', ha='center', va='bottom')

plt.xlabel('Line of Business')
plt.ylabel('% of Total')
plt.xticks(indices, tmp.index)
plt.ylim(0, 100)
plt.grid(axis='y', alpha=0.25)
plt.legend(loc='upper left')
fig.patch.set_facecolor('white')
plt.show()

# Campaign-Level Analysis
exhibition_campaigns = (
    sem_cons[sem_cons['lob']=='Exhibition ']
    .groupby('campaign')
    .agg({'cost': 'sum', 'clicks': 'sum'})
)
exhibition_campaigns['clicks_p_cost'] = exhibition_campaigns['clicks'] / exhibition_campaigns['cost']
top_campaigns = exhibition_campaigns.sort_values(by='clicks_p_cost', ascending=False).head(10)
top_campaigns

**NOTE:** All spend in Exhibition is `EXH_B2C_G&E_National_CA_CONS_NON_EN_S CEA EXH 001_CP2K790`

In [0]:
# Data Preparation
tmp = sem_cons[~sem_cons['lob'].isin(['TRR', 'PDM'])].copy()
tmp = tmp[['week', 'impr.', 'clicks']].groupby('week').sum()
tmp.index = pd.to_datetime(tmp.index)

colors = [palette[3] if tmp['clicks'][i]>0.6*tmp['impr.'][i] else palette[0] for i in range(len(tmp))]

# Visualization
x, y = np.linspace(0, 10**5, 1000), np.linspace(0, 5500, 1000)
fig = plt.figure(figsize=(8, 6), dpi=240)
plt.scatter(tmp['impr.'], tmp['clicks'], color=colors, s=3, label='Week')
plt.plot(x, y, color=palette[2], linestyle='dashed', label='Line of 5.5% CTR')
plt.fill_between(x, y, color=palette[2], alpha=0.25)
plt.gca().xaxis.set_major_formatter(mtick.EngFormatter(unit=''))
plt.gca().yaxis.set_major_formatter(mtick.EngFormatter(unit=''))
plt.xlim(0, 10**5)
plt.ylim(0, 4.4 * 10**4)
plt.xlabel('Impressions')
plt.ylabel('Clicks')
plt.legend()
plt.grid(alpha=0.25)
fig.patch.set_facecolor('white')
plt.show()

# # Identify Inefficient Weeks
# inefficient_weeks = tmp[tmp['clicks'] < 0.055*tmp['impr.']].index.tolist()

# # Campaign-Level Analysis for Inefficient Weeks
# tmp = sem_cons[~sem_cons['lob'].isin(['TRR', 'PDM'])].copy()
# tmp['week'] = pd.to_datetime(tmp['week'])
# tmp = tmp[tmp['week'].isin(inefficient_weeks)]
# tmp = tmp.groupby('campaign').agg({'impr.': 'sum', 'clicks': 'sum'})
# tmp['ctr'] = tmp['clicks'] / tmp['impr.']
# low_performers = tmp.sort_values(by='ctr', ascending=True).head(10)
# low_performers

# Identify Efficient Weeks
efficient_weeks = tmp[tmp['clicks'] > 0.5*tmp['impr.']].index.tolist()

# Campaign-Level Analysis for Efficient Weeks
tmp = sem_cons[~sem_cons['lob'].isin(['TRR', 'PDM'])].copy()
tmp['week'] = pd.to_datetime(tmp['week'])
tmp = tmp[tmp['week'].isin(efficient_weeks)]
tmp = tmp.groupby('campaign').agg({'impr.': 'sum', 'clicks': 'sum'})
tmp['ctr'] = tmp['clicks'] / tmp['impr.']
low_performers = tmp.sort_values(by='ctr', ascending=True).head(10)
low_performers

#### Conversion Only

In [0]:
sem_cvn = sem[sem['campaign'].str.contains('_CVN_|Conversion|001_CP2K790')].copy()

In [0]:
# Data Preparation
tmp = sem_cvn.copy()
tmp = tmp[~tmp['lob'].isin(['TRR', 'PDM'])]
tmp['lob'] = np.where(tmp['campaign'].str.contains('_AO_|Always On'), 'Exhibition AO', tmp['lob'])
tmp['lob'] = np.where(tmp['lob'].isin(['Exhibition ', 'Exhibition AO', 'IMAX', 'Cineclub']), tmp['lob'], 'Other')
tmp = tmp.groupby('lob').agg({'cost': 'sum', 'conversions': 'sum'}).sort_values(by='cost', ascending=False)
tmp = tmp / tmp.sum() * 100

# Visualization
indices = np.arange(len(tmp))
fig = plt.figure(figsize=(8, 6), dpi=240)

bars1 = plt.bar(indices - 0.15, tmp['cost'], width=0.3, color=palette[0], zorder=2, label='Spend (% of Total)')
bars2 = plt.bar(indices + 0.15, tmp['conversions'], width=0.3, color=palette[1], zorder=2, label='Attendance Conversions (% of Total)')

for bar in bars1 + bars2:
    yval = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2.0, yval + 1, f'{yval:.0f}%', ha='center', va='bottom')

plt.xlabel('Line of Business')
plt.ylabel('% of Total')
plt.xticks(indices, tmp.index)
plt.ylim(0, 100)
plt.grid(axis='y', alpha=0.25)
plt.legend()
fig.patch.set_facecolor('white')
plt.show()

# Campaign-Level Analysis for 'Exhibition '
exhibition_campaigns = (
    sem_cvn[sem_cvn['lob'] == 'Exhibition ']
    .groupby('campaign')
    .agg({'cost': 'sum', 'conversions': 'sum'})
)
exhibition_campaigns['conv_p_cost'] = exhibition_campaigns['conversions'] / exhibition_campaigns['cost']
top_campaigns = exhibition_campaigns.sort_values(by='conv_p_cost', ascending=False).head(10)
top_campaigns

In [0]:
# Data Preparation
tmp = sem_cvn[~sem_cvn['lob'].isin(['TRR', 'PDM'])].copy()
tmp = tmp[['week', 'cost', 'conversions']].groupby('week').sum()
tmp.index = pd.to_datetime(tmp.index)

colors = [palette[3] if tmp['conversions'][i]>6*tmp['cost'][i] else palette[0] for i in range(len(tmp))]

# Visualization
x, y = np.linspace(0, 26000, 1000), np.linspace(0, 26000, 1000)
fig = plt.figure(figsize=(8, 6), dpi=240)
plt.scatter(tmp['cost'], tmp['conversions'], color=colors, s=5, label='Week')
plt.plot(x, y, color=palette[2], linestyle='dashed', label='Line of Equality')
plt.fill_between(x, y, color=palette[2], alpha=0.25)
plt.gca().xaxis.set_major_formatter(mtick.EngFormatter(unit=''))
plt.gca().yaxis.set_major_formatter(mtick.EngFormatter(unit=''))
plt.xlim(0, 26000)
plt.ylim(0, 65000)
plt.xlabel('Weekly Spend')
plt.ylabel('Conversions')
plt.legend()
plt.grid(alpha=0.25)
fig.patch.set_facecolor('white')
plt.show()

# Identify Inefficient Weeks
inefficient_weeks = tmp[tmp['conversions'] < tmp['cost']].index.tolist()

# Campaign-Level Analysis for Inefficient Weeks
tmp = sem_cvn[~sem_cvn['lob'].isin(['TRR', 'PDM'])].copy()
tmp['week'] = pd.to_datetime(tmp['week'])
tmp = tmp[tmp['week'].isin(inefficient_weeks)]
tmp = tmp.groupby('campaign').agg({'cost': 'sum', 'conversions': 'sum'})
tmp['conv_p_cost'] = tmp['conversions'] / tmp['cost']
top_campaigns = tmp.sort_values(by='conv_p_cost', ascending=False).head(10)
top_campaigns

In [0]:
# Data Preparation
tmp = sem_cvn[~sem_cvn['lob'].isin(['TRR', 'PDM'])].copy()
tmp = tmp.groupby('week')[['cost', 'online_ticket_purchases']].sum()

# Visualization
fig = plt.figure(figsize=(8, 6), dpi=240)
plt.bar(tmp.index, tmp['cost'], color='#E5C224', width=5, label='Spend')
plt.plot(tmp['online_ticket_purchases'], color='#065CB1', label='Ticket Purchases from SEM')
plt.ylim(0, 60000)
plt.gca().yaxis.set_major_formatter(mtick.EngFormatter(unit=''))
plt.legend(loc='upper left')
plt.xlabel('Week')
plt.ylabel('Spend')
plt.grid(axis='y', alpha=0.25)
fig.patch.set_facecolor('white')
plt.show()

# Printing correlation
print(np.corrcoef(tmp['cost'], tmp['online_ticket_purchases'])[0][1])

#### Journey

In [0]:
tmp = sem[~sem['lob'].isin(['TRR', 'PDM'])].copy()

impressions = tmp['impr.'].sum()
clicks = tmp['clicks'].sum()
conversions = tmp['conversions'].sum()

labels = ['Impressions', 'Clicks', 'Conversions']
vals = [impressions, clicks, conversions]

df = pd.DataFrame({"Labels": labels, "Values": vals})

fig = plt.figure(figsize=(7, 6), dpi=240)

plt.fill_betweenx(y=[1, 3.8], x1=[10,12], x2=[8,6], color=palette[2]);
plt.fill_betweenx(y=[4, 6.8], x1=[12,14], x2=[6,4], color=palette[1]);
plt.fill_betweenx(y=[7, 9.8], x1=[14,16], x2=[4,2], color=palette[0]);

plt.xticks([],[]);
plt.yticks([2,5,8], df["Labels"][::-1]);

for y, value in zip([2,5,8], np.round(df["Values"][::-1], -3)):
    plt.text(9, y+0.25, f'{value:,.0f}', fontsize=11, fontweight="bold", color="white", ha="center");

fig.patch.set_facecolor('white')

### Meta Spend

#### Awareness Only

In [0]:
meta_awa = meta[meta['objective']=='Awareness'].copy()

In [0]:
# Data Preparation
tmp = meta_awa.copy()
map = {'BRA': 'Brand', 'FLM': 'Event', 'TBR': 'Preopening', 'TFB': 'F&B'}
tmp['lob'] = tmp['lob'].map(map)
tmp = tmp.groupby('lob').agg({'amount_spent_(cad)': 'sum', 'impressions': 'sum'}).sort_values(by='amount_spent_(cad)', ascending=False)
tmp = tmp / tmp.sum() * 100

# Visualization
indices = np.arange(len(tmp))
fig = plt.figure(figsize=(8, 6), dpi=240)

bars1 = plt.bar(indices - 0.15, tmp['amount_spent_(cad)'], width=0.3, color=palette[0], zorder=2, label='Spend (% of Total)')
bars2 = plt.bar(indices + 0.15, tmp['impressions'], width=0.3, color=palette[1], zorder=2, label='Impressions (% of Total)')

for bar in bars1 + bars2:
    yval = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2.0, yval + 1, f'{math.ceil(yval):.0f}%', ha='center', va='bottom')

plt.xlabel('Line of Business')
plt.ylabel('% of Total')
plt.xticks(indices, tmp.index)
plt.ylim(0, 100)
plt.grid(axis='y', alpha=0.25)
plt.legend()
fig.patch.set_facecolor('white')
plt.show()

#### Considerations Only

In [0]:
meta_cons = meta[meta['objective']=='Consideration'].copy()

In [0]:
from statsmodels.nonparametric.smoothers_lowess import lowess

tmp = meta_cons.groupby('week').agg({'amount_spent_(cad)': 'sum', 'link_clicks': 'sum'}).copy()
tmp['cpc'] = tmp['amount_spent_(cad)'] / tmp['link_clicks']
tmp.index = pd.to_datetime(tmp.index)

loess_smoothed = lowess(tmp['cpc'], tmp.index, frac=0.2)
loess_df = pd.DataFrame(loess_smoothed, columns=['x', 'y'])

fig = plt.figure(figsize=(8, 6), dpi=240)
plt.plot(tmp['cpc'], color=palette[0], label='Cost Per Click (CAD)')
plt.plot(tmp.index, loess_df['y'], color=palette[1], linestyle='dashed', label='Trendline')
plt.ylim(0, 2.5)
plt.xlabel('Week')
plt.ylabel('Cost Per Click (CAD)')
plt.legend(loc='upper left')
plt.grid(axis='y', alpha=0.25)
fig.patch.set_facecolor('white')
plt.show()

In [0]:
tmp = meta_cons.copy()
tmp = tmp[~tmp['lob'].isin(['TGP', 'TBR'])]
tmp['lob'] = np.where(tmp['lob'].isin(['BRA', 'FLM', 'PPP', 'GRP']), tmp['lob'], 'Other')
map = {'BRA': 'Brand', 'FLM': 'Film', 'PPP': 'Prepaid', 'GRP': 'G&E', 'Other': 'Other'}
tmp['lob'] = tmp['lob'].map(map)
tmp = tmp.groupby(['lob'])[['amount_spent_(cad)', 'link_clicks']].sum().sort_values(by='amount_spent_(cad)', ascending=False)
tmp = tmp / tmp.sum() * 100

indices = np.arange(len(tmp))
fig = plt.figure(figsize=(8, 6), dpi=240)
bars1 = plt.bar(indices - 0.15, tmp['amount_spent_(cad)'], width=0.3, color='#065CB1', zorder=2, label='Spend (% of Total)')
bars2 = plt.bar(indices + 0.15, tmp['link_clicks'], width=0.3, color='#E5C224', zorder=2, label='Clicks (% of Total)')

for bar in bars1 + bars2:
    yval = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2.0, yval + 1, f'{math.ceil(yval):.0f}%', ha='center', va='bottom', fontsize=10)

plt.xlabel('Line of Business')
plt.ylabel('% of Total')
plt.xticks(indices, tmp.index)
plt.ylim(0, 100)
plt.grid(axis='y', alpha=0.25)
plt.legend()
fig.patch.set_facecolor('white')
plt.show()

#### Conversion Only

In [0]:
meta_exh_new = pd.read_excel(f'{directory}/raw_data/META EXH Mar 2023 to June 2025 weekly report_v2.xlsx')

# Clean similar to `marketing_mix_modelling_exh`
meta_exh_new.columns = meta_exh_new.columns.str.lower().str.replace(' ', '_')
meta_exh_new['reporting_starts'] = pd.to_datetime(meta_exh_new['reporting_starts'])
meta_exh_new['week'] = meta_exh_new['reporting_starts'].dt.to_period('W-THU').dt.start_time
meta_exh_new = meta_exh_new.drop('reporting_starts', axis=1)
meta_exh_new = meta_exh_new.dropna(subset='amount_spent_(cad)')

# Extract lob and objective from campaign or result names
meta_exh_new['lob'] = meta_exh_new['campaign_name'].str.extract(r'(?:CEC|CEA)_(.*?)_')
cases = [meta_exh_new['result_indicator'].str.contains('reach', na=False), 
        meta_exh_new['result_indicator'].str.contains('link_click|landing_page_view', na=False),
        meta_exh_new['result_indicator'].str.contains('offsite_conversion|subscribe_website', na=False)]
meta_exh_new['objective'] = np.select(cases, ['Awareness', 'Consideration', 'Conversion'], default=np.nan)

In [0]:
# Create temporary dataframe
tmp = meta_exh_new[meta_exh_new['objective']=='Conversion'].copy()
tmp['lob'] = np.where(tmp['lob'].isin(['FLM', 'BRA', 'EXH']), tmp['lob'], 'Other')
map = {'BRA': 'Brand', 'FLM': 'Film', 'EXH': 'Exhibition', 'Other': 'Other'}
tmp['lob'] = tmp['lob'].map(map)
tmp = tmp.groupby('lob').agg({'amount_spent_(cad)': 'sum', 'purchase_revenue': 'sum'}).sort_values(by='amount_spent_(cad)', ascending=False)
tmp = tmp / tmp.sum() * 100

# Visualization
indices = np.arange(len(tmp))
fig = plt.figure(figsize=(8, 6), dpi=240)

bars1 = plt.bar(indices - 0.15, tmp['amount_spent_(cad)'], width=0.3, color=palette[0], zorder=2, label='Spend (% of Total)')
bars2 = plt.bar(indices + 0.15, tmp['purchase_revenue'], width=0.3, color=palette[1], zorder=2, label='Revenue (% of Total)')

for bar in bars1 + bars2:
    yval = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2.0, yval + 1, f'{math.ceil(yval):.0f}%', ha='center', va='bottom')

plt.xlabel('Line of Business')
plt.ylabel('% of Total')
plt.xticks(indices, tmp.index)
plt.ylim(0, 100)
plt.grid(axis='y', alpha=0.25)
plt.legend()
fig.patch.set_facecolor('white')
plt.show()

#### Journey

### Snapchat

#### Awareness Only

In [0]:
snap_awa = social[(social['objective']=='Awareness')*(social['publisher']=='Snapchat')].copy()

**NOTE:** No exhibition data for awareness campaigns in analysis period (Q2 2023 to Q1 2025)

#### Consideration Only

In [0]:
snap_cons = social[(social['objective']=='Consideration')*(social['publisher']=='Snapchat')].copy()

**NOTE:** No exhibition data for consideration campaigns in analysis period (Q2 2023 to Q1 2025)

#### Conversion Only

In [0]:
snap_cvn = social[(social['objective']=='Conversion')*(social['publisher']=='Snapchat')].copy()

In [0]:
tmp = snap_cvn.copy()
# tmp['lob'] = np.where(tmp['lob'].isin(['BRA', 'FLM', 'PPP', 'GRP']), tmp['lob'], 'Other')
# map = {'BRA': 'Brand', 'FLM': 'Film', 'PPP': 'Prepaid', 'GRP': 'G&E', 'Other': 'Other'}
# tmp['lob'] = tmp['lob'].map(map)
tmp = tmp.groupby(['line_of_business'])[['investment', 'purchase_revenue']].sum().sort_values(by='investment', ascending=False)
tmp = tmp / tmp.sum() * 100

indices = np.arange(len(tmp))
fig = plt.figure(figsize=(8, 6), dpi=240)
bars1 = plt.bar(indices - 0.15, tmp['investment'], width=0.3, color='#065CB1', zorder=2, label='Spend (% of Total)')
bars2 = plt.bar(indices + 0.15, tmp['purchase_revenue'], width=0.3, color='#E5C224', zorder=2, label='Revenue (% of Total)')

for bar in bars1 + bars2:
    yval = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2.0, yval + 1, f'{yval:.0f}%', ha='center', va='bottom', fontsize=10)

plt.xlabel('Line of Business')
plt.ylabel('% of Total')
plt.xticks(indices, tmp.index)
plt.ylim(0, 100)
plt.grid(axis='y', alpha=0.25)
plt.legend()
fig.patch.set_facecolor('white')
plt.show()

In [0]:
tmp = snap_cvn.copy()
tmp = tmp[['campaign_name', 'investment', 'purchase_revenue']].groupby(['campaign_name']).sum()
tmp['roas'] = tmp['purchase_revenue']/tmp['investment']
tmp = tmp.sort_values(by='roas')[-10:].reset_index()
map = {'Spiderman: Across The Spider Verse': 'Spiderman: Across \nThe Spider Verse',
       'Mission: Impossible Dead Reckoning - Part One': 'Mission: Impossible Dead \nReckoning - Part One',
       'Indiana Jones and the Dial of Destiny': 'Indiana Jones and the \nDial of Destiny'}
tmp['campaign_name'] = tmp['campaign_name'].replace(map)

colors = ['#E5C224' if i=='Film Always On' else '#065CB1' for i in tmp['campaign_name']]

fig = plt.figure(figsize=(6, 6), dpi=240)
plt.barh(y=tmp['campaign_name'], width=tmp['roas'], color=colors, zorder=2)
plt.gca().xaxis.set_major_formatter(mtick.EngFormatter(unit=''))
plt.xlabel('ROAS')
plt.grid(axis='x', alpha=0.25)
fig.patch.set_facecolor('white')
plt.show()

In [0]:
tmp = snap_cvn.copy()
tmp = tmp[['campaign_name', 'investment', 'purchase_revenue']].groupby(['campaign_name']).sum()
tmp['roas'] = tmp['purchase_revenue']/tmp['investment']
tmp = tmp.sort_values(by='roas', ascending=False)[-10:].reset_index()
map = {'Aquaman and the Lost Kingdom': 'Aquaman and \nthe Lost Kingdom',
       'Transformers: Rise of the Beasts': 'Transformers: Rise of \nthe Beasts'}
tmp['campaign_name'] = tmp['campaign_name'].replace(map)

colors = ['#E5C224' if i=='Film Always On' else '#065CB1' for i in tmp['campaign_name']]

fig = plt.figure(figsize=(6, 6), dpi=240)
plt.barh(y=tmp['campaign_name'], width=tmp['roas'], color=colors, zorder=2)
plt.gca().xaxis.set_major_formatter(mtick.EngFormatter(unit=''))
plt.xlabel('ROAS')
plt.grid(axis='x', alpha=0.25)
fig.patch.set_facecolor('white')
plt.show()

### TikTok

#### Awareness Only

#### Consideration Only

In [0]:
tik_cons = social[(social['objective']=='Consideration')*(social['publisher']=='TikTok')]

In [0]:
tmp = tik_cons.copy()
tmp = tmp[tmp['line_of_business']!='LBE Brand']
# tmp['lob'] = np.where(tmp['lob'].isin(['BRA', 'FLM', 'PPP', 'GRP']), tmp['lob'], 'Other')
# map = {'BRA': 'Brand', 'FLM': 'Film', 'PPP': 'Prepaid', 'GRP': 'G&E', 'Other': 'Other'}
# tmp['lob'] = tmp['lob'].map(map)
tmp = tmp.groupby(['line_of_business'])[['investment', 'clicks']].sum().sort_values(by='investment', ascending=False)
tmp = tmp / tmp.sum() * 100

indices = np.arange(len(tmp))
fig = plt.figure(figsize=(8, 6), dpi=240)
bars1 = plt.bar(indices - 0.15, tmp['investment'], width=0.3, color='#065CB1', zorder=2, label='Spend (% of Total)')
bars2 = plt.bar(indices + 0.15, tmp['clicks'], width=0.3, color='#E5C224', zorder=2, label='Clicks (% of Total)')

for bar in bars1 + bars2:
    yval = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2.0, yval + 1, f'{yval:.0f}%', ha='center', va='bottom', fontsize=10)

plt.xlabel('Line of Business')
plt.ylabel('% of Total')
plt.xticks(indices, tmp.index)
plt.ylim(0, 100)
plt.grid(axis='y', alpha=0.25)
plt.legend()
fig.patch.set_facecolor('white')
plt.show()

#### Conversion Only

In [0]:
tik_cvn = social[(social['objective']=='Conversion')*(social['publisher']=='TikTok')]

In [0]:
tmp = tik_cvn.groupby('week').agg({'investment': 'sum', 'purchase_revenue': 'sum'}).reset_index()

fig = plt.figure(figsize=(8, 6), dpi=240)

x, y = np.linspace(0, 18000, 1000), np.linspace(0, 180000, 1000)

colors = [palette[3] if tmp['purchase_revenue'][i]<10*tmp['investment'][i] else palette[0] for i in range(len(tmp))]

plt.scatter(tmp['investment'], tmp['purchase_revenue'], color=colors, s=5, zorder=2, label='Week')
plt.plot(x, y, color=palette[2], linestyle='dashed', label='ROAS = 10')
plt.fill_between(x, y, color=palette[2], alpha=0.25)
plt.xlabel('Spend ($)')
plt.ylabel('Purchase Revenue ($)')
plt.xlim(0, 1.8*10**4)
plt.ylim(0, 1.8*10**5)
plt.gca().xaxis.set_major_formatter(mtick.EngFormatter(unit=''))
plt.gca().yaxis.set_major_formatter(mtick.EngFormatter(unit=''))
plt.grid(alpha=0.25)
plt.legend(loc='upper left')
fig.patch.set_facecolor('white')
plt.show()

print('Correlation =', np.corrcoef(tmp['investment'], tmp['purchase_revenue'])[0][1])

In [0]:
tmp = tik_cvn.copy()
tmp = tmp[['campaign_name', 'investment', 'purchase_revenue']].groupby(['campaign_name']).sum()
tmp['roas'] = tmp['purchase_revenue']/tmp['investment']
tmp = tmp.sort_values(by='roas')[-10:].reset_index()
map = {'Spiderman: Across The Spider Verse': 'Spiderman: Across \nThe Spider Verse',
       'Dungeons & Dragons: Honor Among Thieves': 'Dungeons & Dragons: \nHonor Among Thieves',
       'Transformers: Rise of the Beasts': 'Transformers: Rise of \nthe Beasts'}
tmp['campaign_name'] = tmp['campaign_name'].replace(map)

colors = colors = ['#E5C224' if i=='Film Always On' else '#065CB1' for i in tmp['campaign_name']]

fig = plt.figure(figsize=(8, 7), dpi=240)
plt.barh(y=tmp['campaign_name'], width=tmp['roas'], color=colors, zorder=2)
plt.gca().xaxis.set_major_formatter(mtick.EngFormatter(unit=''))
plt.xlabel('ROAS')
plt.grid(axis='x', alpha=0.25)
fig.patch.set_facecolor('white')
plt.show()

In [0]:
tmp = tik_cvn.copy()
tmp = tmp[['campaign_name', 'investment', 'purchase_revenue']].groupby(['campaign_name']).sum()
tmp['roas'] = tmp['purchase_revenue']/tmp['investment']
tmp = tmp.sort_values(by='roas', ascending=False)[-10:].reset_index()
map = {'Spiderman: Across The Spider Verse': 'Spiderman: Across \nThe Spider Verse',
       'Indiana Jones and the Dial of Destiny': 'Indiana Jones and the \nDial of Destiny',
       'Transformers: Rise of the Beasts': 'Transformers: Rise of \nthe Beasts'}
tmp['campaign_name'] = tmp['campaign_name'].replace(map)

colors = colors = ['#E5C224' if i=='Film Always On' else '#065CB1' for i in tmp['campaign_name']]

fig = plt.figure(figsize=(8, 7), dpi=240)
plt.barh(y=tmp['campaign_name'], width=tmp['roas'], color=colors, zorder=2)
plt.gca().xaxis.set_major_formatter(mtick.EngFormatter(unit=''))
plt.xlabel('ROAS')
plt.grid(axis='x', alpha=0.25)
fig.patch.set_facecolor('white')
plt.show()

### Programmatic

#### Awareness Only

#### Consideration Only

In [0]:
prog_cons = programmatic[programmatic['objective']=='Consideration'].copy()

In [0]:
tmp = prog_cons.copy()
tmp = tmp[~tmp['line_of_business'].isin(['LBE Brand', 'Performance'])]
tmp = tmp.groupby(['line_of_business'])[['investment', 'clicks']].sum().sort_values(by='investment', ascending=False)
tmp = tmp / tmp.sum() * 100

indices = np.arange(len(tmp))
fig = plt.figure(figsize=(8, 6), dpi=240)
bars1 = plt.bar(indices - 0.15, tmp['investment'], width=0.3, color='#065CB1', zorder=2, label='Spend (% of Total)')
bars2 = plt.bar(indices + 0.15, tmp['clicks'], width=0.3, color='#E5C224', zorder=2, label='Clicks (% of Total)')

for bar in bars1 + bars2:
    yval = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2.0, yval + 1, f'{yval:.0f}%', ha='center', va='bottom', fontsize=10)

plt.xlabel('Line of Business')
plt.ylabel('% of Total')
plt.xticks(indices, tmp.index)
plt.ylim(0, 100)
plt.grid(axis='y', alpha=0.25)
plt.legend()
fig.patch.set_facecolor('white')
plt.show()

#### Conversion Only

In [0]:
prog_cvn = programmatic[programmatic['objective']=='Conversion']

In [0]:
tmp = prog_cvn.copy()
tmp = tmp.groupby(['line_of_business'])[['investment', 'purchase_revenue']].sum().sort_values(by='investment', ascending=False)
tmp = tmp / tmp.sum() * 100

indices = np.arange(len(tmp))
fig = plt.figure(figsize=(8, 6), dpi=240)
bars1 = plt.bar(indices - 0.15, tmp['investment'], width=0.3, color='#065CB1', zorder=2, label='Spend (% of Total)')
bars2 = plt.bar(indices + 0.15, tmp['purchase_revenue'], width=0.3, color='#E5C224', zorder=2, label='Revenue (% of Total)')

for bar in bars1 + bars2:
    yval = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2.0, yval + 1, f'{yval:.0f}%', ha='center', va='bottom', fontsize=10)

plt.xlabel('Line of Business')
plt.ylabel('% of Total')
plt.xticks(indices, tmp.index)
plt.ylim(0, 100)
plt.grid(axis='y', alpha=0.25)
plt.legend()
fig.patch.set_facecolor('white')
plt.show()

In [0]:
tmp = prog_cvn.copy()
tmp = tmp[['campaign_name', 'investment', 'purchase_revenue']].groupby(['campaign_name']).sum()
tmp['roas'] = tmp['purchase_revenue']/tmp['investment']
tmp.index = tmp.index.str.replace(' - Outcomes', '')
tmp = tmp.sort_values(by='roas').reset_index()
map = {'Godzilla x Kong: The New Empire - IMAX': 'Godzilla x Kong: The \nNew Empire - IMAX',
       'Guardians of the Galaxy Vol. 3 IMAX': 'Guardians of the Galaxy \nVol. 3 IMAX',
       'Convert & Upgrade Co-Branded': 'Performance Campaigns'}
tmp['campaign_name'] = tmp['campaign_name'].replace(map)

colors = [palette[1] if i=='Performance Campaigns' else palette[0] for i in tmp['campaign_name']]

fig = plt.figure(figsize=(8, 7), dpi=240)
plt.barh(y=tmp['campaign_name'], width=tmp['roas'], color=colors, zorder=2)
plt.gca().xaxis.set_major_formatter(mtick.EngFormatter(unit=''))
plt.xlabel('ROAS')
plt.grid(axis='x', alpha=0.25)
fig.patch.set_facecolor('white')
plt.show()

#### Journey

In [0]:
tmp = programmatic[programmatic['objective'].isin(['Awareness', 'Consideration', 'Conversion'])].copy()

impressions = tmp['impressions'].sum()
clicks = tmp['clicks'].sum()
conversions = tmp['purchases'].sum()

labels = ['Impressions', 'Clicks', 'Conversions']
vals = [impressions, clicks, conversions]

df = pd.DataFrame({"Labels": labels, "Values": vals})

fig = plt.figure(figsize=(7, 6), dpi=240)

plt.fill_betweenx(y=[1, 3.8], x1=[10,12], x2=[8,6], color=palette[2]);
plt.fill_betweenx(y=[4, 6.8], x1=[12,14], x2=[6,4], color=palette[1]);
plt.fill_betweenx(y=[7, 9.8], x1=[14,16], x2=[4,2], color=palette[0]);

plt.xticks([],[]);
plt.yticks([2,5,8], df["Labels"][::-1]);

for y, value in zip([2,5,8], np.round(df["Values"][::-1], -3)):
    plt.text(9, y+0.25, f'{value:,.0f}', fontsize=11, fontweight="bold", color="white", ha="center");

fig.patch.set_facecolor('white')

### Email Campaigns

#### Awareness Only

In [0]:
tmp = email.copy()

conditions = [
    (tmp['journey_name'].str.contains('STO')),
    (tmp['journey_name'].str.contains('CineClub|Cine Club')),
    (tmp['journey_name'].str.contains('MKTB|MKBT|MKTX|Scene+|Tuesday|Friday')),
    (tmp['journey_name'].str.contains('MKTT|EVCN|MKTI|Production'))
]
choices = ['Store', 'CineClub', 'Brand', 'Film']
tmp['lob'] = np.where(tmp['lob'].isna(), np.select(conditions, choices, default=tmp['lob']), tmp['lob'])

# Data Preparation
tmp = tmp[~tmp['lob'].isin(['LBE', np.NaN])]
tmp['lob'] = np.where(tmp['lob']=='CineClub', 'Cineclub', tmp['lob'])
tmp['lob'] = np.where(tmp['lob'].isin(['Film', 'Brand', 'Cineclub']), tmp['lob'], 'Other')
tmp = tmp.groupby('lob').agg({'sent': 'sum', 'unique_opens': 'sum', 'unique_clicks': 'sum'}).sort_values(by='sent', ascending=False)
tmp = tmp / tmp.sum() * 100

# Visualization
indices = np.arange(len(tmp))
fig = plt.figure(figsize=(8, 6), dpi=240)

bars1 = plt.bar(indices - 0.15, tmp['sent'], width=0.3, color=palette[0], zorder=2, label='Sends (% of Total)')
bars2 = plt.bar(indices + 0.15, tmp['unique_opens'], width=0.3, color=palette[1], zorder=2, label='Unique Opens (% of Total)')

for bar in bars1 + bars2:
    yval = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2.0, yval + 1, f'{yval:.0f}%', ha='center', va='bottom')

plt.xlabel('Line of Business')
plt.ylabel('% of Total')
plt.xticks(indices, tmp.index)
plt.ylim(0, 100)
plt.grid(axis='y', alpha=0.25)
plt.legend()
fig.patch.set_facecolor('white')
plt.show()

#### Consideration Only

In [0]:
tmp = email.copy()

conditions = [
    (tmp['journey_name'].str.contains('STO')),
    (tmp['journey_name'].str.contains('CineClub|Cine Club')),
    (tmp['journey_name'].str.contains('MKTB|MKBT|MKTX|Scene+|Tuesday|Friday')),
    (tmp['journey_name'].str.contains('MKTT|EVCN|MKTI|Production'))
]
choices = ['Store', 'CineClub', 'Brand', 'Film']
tmp['lob'] = np.where(tmp['lob'].isna(), np.select(conditions, choices, default=tmp['lob']), tmp['lob'])

# Data Preparation
tmp = tmp[~tmp['lob'].isin(['LBE', np.NaN])]
tmp['lob'] = np.where(tmp['lob']=='CineClub', 'Cineclub', tmp['lob'])
tmp['lob'] = np.where(tmp['lob'].isin(['Film', 'Brand', 'Cineclub']), tmp['lob'], 'Other')
tmp = tmp.groupby('lob').agg({'sent': 'sum', 'unique_opens': 'sum', 'unique_clicks': 'sum'}).sort_values(by='sent', ascending=False)
tmp = tmp / tmp.sum() * 100

# Visualization
indices = np.arange(len(tmp))
fig = plt.figure(figsize=(8, 6), dpi=240)

bars1 = plt.bar(indices - 0.15, tmp['sent'], width=0.3, color=palette[0], zorder=2, label='Sends (% of Total)')
bars2 = plt.bar(indices + 0.15, tmp['unique_clicks'], width=0.3, color=palette[1], zorder=2, label='Unique Clicks (% of Total)')

for bar in bars1 + bars2:
    yval = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2.0, yval + 1, f'{yval:.0f}%', ha='center', va='bottom')

plt.xlabel('Line of Business')
plt.ylabel('% of Total')
plt.xticks(indices, tmp.index)
plt.ylim(0, 100)
plt.grid(axis='y', alpha=0.25)
plt.legend()
fig.patch.set_facecolor('white')
plt.show()

#### Conversion Only

In [0]:
email_app_revenue = pd.read_csv(f'{directory}/raw_data/Email Ticketing & Revenue by Week - App.csv', skiprows=12)
email_web_revenue = pd.read_csv(f'{directory}/raw_data/Email Ticketing & Revenue by Week - Web.csv', skiprows=12)

In [0]:
email_app_revenue.columns = ['week', 'purchases1', 'purchases2', 'revenue1', 'revenue2']
email_web_revenue.columns = ['week', 'purchases1', 'purchases2', 'revenue1', 'revenue2']

email_revenue = pd.concat([email_app_revenue, email_web_revenue.drop('week', axis=1)], axis=1)

email_revenue['purchases'] = email_revenue.filter(like='purchases').sum(axis=1)
email_revenue['revenue'] = email_revenue.filter(like='revenue').sum(axis=1)

email_revenue['week'] = pd.to_datetime(email_revenue['week'])
email_revenue['week'] = email_revenue['week'].dt.to_period('W-THU').dt.start_time

In [0]:
tmp1 = email[email['week'].between('2023-03-31', '2025-03-31')].copy()
tmp1 = tmp1.groupby('week').agg({'sent': 'sum'})

tmp2 = email_revenue[email_revenue['week'].between('2023-03-31', '2025-03-31')].copy()
tmp2 = tmp2.groupby('week').agg({'revenue': 'sum'})

tmp = pd.merge(tmp1, tmp2, left_on='week', right_on='week').fillna(0)

fig = plt.figure(figsize=(8, 6), dpi=240)
plt.scatter(tmp['sent'], tmp['revenue'], color=palette[0], s=5, label='Week', zorder=2)
plt.gca().xaxis.set_major_formatter(mtick.EngFormatter(unit=''))
plt.gca().yaxis.set_major_formatter(mtick.EngFormatter(unit=''))
plt.xlim(0, 3*10**7)
plt.ylim(0, 6*10**5)
plt.xlabel('Email Sends')
plt.ylabel('Revenue')
plt.legend()
plt.grid(alpha=0.25)
plt.show()

#### Journey

In [0]:
tmp = email.copy()

impressions = tmp['unique_opens'].sum()
clicks = tmp['unique_clicks'].sum()
conversions = email_revenue['purchases'].copy().sum()

labels = ['Impressions', 'Clicks', 'Conversions']
vals = [impressions, clicks, conversions]

df = pd.DataFrame({"Labels": labels, "Values": vals})

fig = plt.figure(figsize=(7, 6), dpi=240)

plt.fill_betweenx(y=[1, 3.8], x1=[10,12], x2=[8,6], color=palette[2]);
plt.fill_betweenx(y=[4, 6.8], x1=[12,14], x2=[6,4], color=palette[1]);
plt.fill_betweenx(y=[7, 9.8], x1=[14,16], x2=[4,2], color=palette[0]);

plt.xticks([],[]);
plt.yticks([2,5,8], df["Labels"][::-1]);

for y, value in zip([2,5,8], np.round(df["Values"][::-1], -3)):
    plt.text(9, y+0.25, f'{value:,.0f}', fontsize=11, fontweight="bold", color="white", ha="center");

fig.patch.set_facecolor('white')

### Cineclub

In [0]:
tmp1 = cineclub.copy()
tmp1.set_index('week', inplace=True)
tmp1['cineclub_signups'] = tmp1.filter(like='signups').sum(axis=1)
tmp1 = tmp1[['cineclub_signups']].groupby(tmp1.index).sum()

tmp2 = sem[sem['lob']=='Cineclub'].groupby('week').agg({'cost': 'sum'}).copy()
tmp3 = meta[meta['lob']=='CCB'].groupby('week').agg({'amount_spent_(cad)': 'sum'}).copy()
tmp4 = programmatic[programmatic['line_of_business']=='Cineclub'].groupby('week').agg({'investment': 'sum'}).copy()

tmp1.index = pd.to_datetime(tmp1.index)
tmp2.index = pd.to_datetime(tmp2.index)
tmp3.index = pd.to_datetime(tmp3.index)
tmp4.index = pd.to_datetime(tmp4.index)

tmp = pd.concat([tmp1, tmp2, tmp3, tmp4], axis=1).fillna(0)
tmp.columns = ['signups', 'sem_spend', 'meta_spend', 'programmatic_spend']
tmp['cineclub_spend'] = tmp[tmp.columns[1:]].sum(axis=1)

In [0]:
fig, ax1 = plt.subplots(figsize=(7, 6), dpi=240)
ax2 = ax1.twinx()
ax1.bar(tmp.index, tmp['sem_spend'], bottom=tmp['meta_spend']+tmp['programmatic_spend'], width=5, color=palette[0], label='SEM', zorder=2)
ax1.bar(tmp.index, tmp['programmatic_spend'], bottom=tmp['meta_spend'], width=5, color=palette[0], alpha=0.7, label='Programmatic', zorder=2)
ax1.bar(tmp.index, tmp['meta_spend'], width=5, color=palette[0], alpha=0.4, label='Meta', zorder=2)
ax2.plot(tmp['signups'], color=palette[1], label='Signups', linewidth=2)
plt.xlabel('Week')
plt.ylabel('Amount')
ax1.set_ylim(0, 10**5)
ax2.set_ylim(0, 10**4)
ax1.set_xlabel('Week')
ax1.set_ylabel('Cineclub Spend ($)')
ax2.set_ylabel('Cineclub Signups')
ax1.yaxis.set_major_formatter(mtick.EngFormatter(unit=''))
ax2.yaxis.set_major_formatter(mtick.EngFormatter(unit=''))
ax1.legend(loc='upper left')
ax2.legend(loc='upper right')
plt.grid(axis='y', alpha=0.25)
fig.patch.set_facecolor('white')
plt.show()

print(np.corrcoef(tmp['cineclub_spend'], tmp['signups'])[0][1])

In [0]:
# Creating a cross-correlation function over lags
def ccf(x: pd.Series, y: pd.Series, max_lag: int = 12):
    lags = range(0, max_lag + 1)
    correlations = [x.corr(y.shift(lag)) for lag in lags]
    return lags, correlations

lags, corr_values = ccf(tmp['cineclub_spend'], tmp['signups'])
fig = plt.figure(figsize=(8, 6), dpi=240)
plt.bar(lags, corr_values, color=[palette[1] if lag<=1 else palette[0] for lag in lags], zorder=2)
plt.xlabel('Weeks after Cineclub Spend')
plt.ylabel('Correlation with Signups')
plt.ylim(-0.2, 0.5)
plt.grid(axis='y', alpha=0.25)
fig.patch.set_facecolor('white')
plt.show()

In [0]:
tmp1 = attendance_and_revenue.copy()
tmp1['week'] = pd.to_datetime(tmp1['week'])
tmp1 = tmp1.groupby('week').agg({'attendance': 'sum'})

tmp2 = cineclub.copy()
tmp2['week'] = pd.to_datetime(tmp2['week'])
tmp2 = tmp2.groupby('week').sum()
tmp2['cineclub_signups'] = tmp2.filter(like='signups').sum(axis=1)

tmp = pd.merge(tmp1, tmp2, on='week')

lags, corr_values = ccf(tmp['cineclub_signups'], tmp['attendance'])
fig = plt.figure(figsize=(8, 6), dpi=240)
plt.bar(lags, corr_values, color=[palette[1] if lag<=1 else palette[0] for lag in lags], zorder=2)
plt.xlabel('Weeks after Cineclub Signup')
plt.ylabel('Correlation with Attendance')
plt.ylim(-0.4, 1)
plt.grid(axis='y', alpha=0.25)
fig.patch.set_facecolor('white')
plt.show()

### IMAX

In [0]:
tmp1 = sem_new[sem_new['account_name']=='EXH - IMAX'][['week', 'cost', 'revenue']].copy()
tmp1['week'] = tmp1['week'].dt.to_period('W-THU').dt.start_time
tmp1 = tmp1[tmp1['week'].between('2023-03-31', '2025-03-31')].groupby('week').sum()
tmp2 = meta_exh_new[meta_exh_new['campaign_name'].str.contains('IMX')][['week', 'amount_spent_(cad)', 'purchase_revenue']].copy()
tmp2 = tmp2[tmp2['week'].between('2023-03-31', '2025-03-31')].groupby('week').sum()
tmp3 = programmatic[programmatic['raw_campaign_name'].str.contains('IMX|IMAX')][['week', 'investment', 'purchase_revenue']].copy()
tmp3 = tmp3[tmp3['week'].between('2023-03-31', '2025-03-31')].groupby('week').sum()
tmp4 = attendance_and_revenue.groupby('week').agg({'attendance': 'sum'}).copy()

tmp = pd.merge(tmp1, tmp2, left_index=True, right_index=True, how='outer').fillna(0)
tmp = pd.merge(tmp, tmp3, left_index=True, right_index=True, how='outer').fillna(0)
tmp = pd.merge(tmp, tmp4, left_index=True, right_index=True, how='outer').fillna(0)

tmp.columns = ['sem_spend', 'sem_revenue', 'meta_spend', 'meta_revenue', 'programmatic_spend', 'programmatic_revenue', 'attendance']
tmp['imax_revenue'] = tmp[['sem_revenue', 'meta_revenue', 'programmatic_revenue']].sum(axis=1)
tmp['imax_spend'] = tmp[['sem_spend', 'meta_spend', 'programmatic_spend']].sum(axis=1)

In [0]:
fig, ax1 = plt.subplots(figsize=(9, 7), dpi=240)
ax2 = ax1.twinx()
ax1.bar(tmp.index, tmp['sem_spend'], bottom=tmp['meta_spend']+tmp['programmatic_spend'], width=5, color=palette[0], label='SEM', zorder=2)
ax1.bar(tmp.index, tmp['programmatic_spend'], bottom=tmp['meta_spend'], width=5, color=palette[0], alpha=0.7, label='Programmatic', zorder=2)
ax1.bar(tmp.index, tmp['meta_spend'], width=5, color=palette[0], alpha=0.4, label='Meta', zorder=2)
ax2.plot(tmp['imax_revenue'], color=palette[1], label='Revenue', linewidth=2)
plt.xlabel('Week')
plt.ylabel('Amount')
ax1.set_ylim(0, 7*10**4)
ax2.set_ylim(0, 7*10**5)
ax1.set_xlabel('Week')
ax1.set_ylabel('IMAX Spend ($)')
ax2.set_ylabel('IMAX Revenue ($)')
ax1.yaxis.set_major_formatter(mtick.EngFormatter(unit=''))
ax2.yaxis.set_major_formatter(mtick.EngFormatter(unit=''))
ax1.legend(loc='upper left')
ax2.legend(loc='upper right')
plt.grid(axis='y', alpha=0.25)
fig.patch.set_facecolor('white')
plt.show()

print(np.corrcoef(tmp.filter(like='spend').sum(axis=1), tmp['imax_revenue'])[0][1])

In [0]:
fig, ax1 = plt.subplots(figsize=(9, 7), dpi=240)
ax2 = ax1.twinx()
ax1.bar(tmp.index, tmp['sem_spend'], bottom=tmp['meta_spend']+tmp['programmatic_spend'], width=5, color=palette[0], label='SEM', zorder=2)
ax1.bar(tmp.index, tmp['programmatic_spend'], bottom=tmp['meta_spend'], width=5, color=palette[0], alpha=0.7, label='Programmatic', zorder=2)
ax1.bar(tmp.index, tmp['meta_spend'], width=5, color=palette[0], alpha=0.4, label='Meta', zorder=2)
ax2.plot(tmp['attendance'], color=palette[1], label='Attendance', linewidth=2)
ax1.set_ylim(0, 6*10**4)
ax2.set_ylim(0, 3*10**6)
ax1.set_xlabel('Week')
ax1.set_ylabel('IMAX Spend ($)')
ax2.set_ylabel('Attendance')
ax1.yaxis.set_major_formatter(mtick.EngFormatter(unit=''))
ax2.yaxis.set_major_formatter(mtick.EngFormatter(unit=''))
ax1.legend(loc='upper left')
ax2.legend(loc='upper right')
plt.grid(axis='y', alpha=0.25)
fig.patch.set_facecolor('white')
plt.show()

print(np.corrcoef(tmp['attendance'], tmp['imax_revenue'])[0][1])

### Summary

In [0]:
# Creating temporary dataframes for each channel
tmp1 = sem_new.copy()
tmp1['lob'] = np.select(
    [tmp1['account_name'].str.contains('Brand'), tmp1['account_name'].str.contains('Cineclub'), 
     tmp1['account_name'].str.contains('Film'), tmp1['account_name'].str.contains('Prepaid'), 
     tmp1['account_name'].str.contains('EXH - Cineplex'), tmp1['account_name'].str.contains('IMAX'), 
     tmp1['account_name'].str.contains('G&E|B2B')], 
    ['Brand', 'Cineclub', 'Film', 'Prepaid', 'Exhibition', 'IMAX', 'G&E'],
    default='Other')
tmp1['objective'] = np.where(
    tmp1['campaign'].str.contains('Conversion|_CVN|_CVS|001_CP2K790', na=False),
    'Conversion',
    'Upper Funnel'
)
tmp1['channel'] = 'SEM'
tmp1 = tmp1[['week', 'objective', 'lob', 'channel', 'campaign', 
             'cost', 'impr.', 'clicks', 'revenue']]
tmp1.columns = ['week', 'objective', 'lob', 'channel', 'campaign_name', 
                'spend', 'impressions', 'clicks', 'revenue']

tmp2 = meta_exh_new.copy()
tmp2['lob'] = np.select(
    [tmp2['lob']=='BRA', tmp2['lob']=='CCB', tmp2['lob']=='FLM', tmp2['lob']=='PPP', tmp2['lob']=='EXH', 
     tmp2['lob']=='IMX', tmp2['lob']=='GRP'], 
    ['Brand', 'Cineclub', 'Film', 'Prepaid', 'Exhibition', 'IMAX', 'G&E'],
    default='Other')
tmp2['objective'] = np.where(tmp2['objective']=='Conversion', 'Conversion', 'Upper Funnel')
tmp2['channel'] = 'Meta'
tmp2 = tmp2[['week', 'objective', 'lob', 'channel', 'campaign_name', 
             'amount_spent_(cad)', 'impressions', 'link_clicks', 'purchase_revenue']]
tmp2.columns = ['week', 'objective', 'lob', 'channel', 'campaign_name', 
                'spend', 'impressions', 'clicks', 'revenue']

tmp3 = social[social['publisher']=='Snapchat'].copy()
map = {'Brand': 'Brand', 'Cineclub': 'Cineclub', 'Film Marketing': 'Film', 'Performance': 'Performance', 
       'LBE Brand': 'Other', 'Gaming': 'Other', 'Enhanced Experiences': 'Enhanced Experiences', 'B2C': 'Prepaid'}
tmp3['line_of_business'] = tmp3['line_of_business'].map(map)
tmp3['objective'] = np.where(tmp3['objective']=='Conversion', 'Conversion', 'Upper Funnel')
tmp3['channel'] = 'Snapchat'
tmp3 = tmp3[['week', 'objective', 'line_of_business', 'channel', 'campaign_name', 
             'investment', 'impressions', 'clicks', 'purchase_revenue']]
tmp3.columns = ['week', 'objective', 'lob', 'channel', 'campaign_name', 
                'spend', 'impressions', 'clicks', 'revenue']

tmp4 = social[social['publisher']=='TikTok'].copy()
map = {'Brand': 'Brand', 'Cineclub': 'Cineclub', 'Film Marketing': 'Film', 'Performance': 'Performance', 
       'LBE Brand': 'Other', 'Gaming': 'Other', 'Enhanced Experiences': 'Enhanced Experiences', 'B2C': 'Prepaid'}
tmp4['line_of_business'] = tmp4['line_of_business'].map(map)
tmp4['objective'] = np.where(tmp4['objective']=='Conversion', 'Conversion', 'Upper Funnel')
tmp4['channel'] = 'Snapchat'
tmp4 = tmp4[['week', 'objective', 'line_of_business', 'channel', 'campaign_name', 
             'investment', 'impressions', 'clicks', 'purchase_revenue']]
tmp4.columns = ['week', 'objective', 'lob', 'channel', 'campaign_name', 
                'spend', 'impressions', 'clicks', 'revenue']

tmp5 = programmatic[programmatic['channel']=='Programmatic Video'].copy()
map = {'Brand': 'Brand', 'Cineclub': 'Cineclub', 'Film Marketing': 'Film', 'Performance': 'Performance', 
       'LBE Brand': 'Other', 'Gaming': 'Other', 'Enhanced Experiences': 'Enhanced Experiences', 'B2C': 'Prepaid'}
tmp5['line_of_business'] = tmp5['line_of_business'].map(map)
tmp5['objective'] = np.where(tmp5['objective']=='Conversion', 'Conversion', 'Upper Funnel')
tmp5['channel'] = 'Programmatic Video'
tmp5 = tmp5[['week', 'objective', 'line_of_business', 'channel', 'campaign_name', 
             'investment', 'impressions', 'clicks', 'purchase_revenue']]
tmp5.columns = ['week', 'objective', 'lob', 'channel', 'campaign_name', 
                'spend', 'impressions', 'clicks', 'revenue']

tmp6 = programmatic[programmatic['channel']=='Programmatic Display'].copy()
map = {'Brand': 'Brand', 'Cineclub': 'Cineclub', 'Film Marketing': 'Film', 'Performance': 'Performance', 
       'LBE Brand': 'Other', 'Gaming': 'Other', 'Enhanced Experiences': 'Enhanced Experiences', 'B2C': 'Prepaid'}
tmp6['line_of_business'] = tmp6['line_of_business'].map(map)
tmp6['objective'] = np.where(tmp6['objective']=='Conversion', 'Conversion', 'Upper Funnel')
tmp6['channel'] = 'Programmatic Display'
tmp6 = tmp6[['week', 'objective', 'line_of_business', 'channel', 'campaign_name', 
             'investment', 'impressions', 'clicks', 'purchase_revenue']]
tmp6.columns = ['week', 'objective', 'lob', 'channel', 'campaign_name', 
                'spend', 'impressions', 'clicks', 'revenue']

# Merging all temporary dataframes
tmp = pd.concat([tmp1, tmp2, tmp3, tmp4, tmp5, tmp6])
tmp['week'] = np.where(tmp['week'].dt.weekday==0, tmp['week'].dt.to_period('W-THU').dt.start_time, tmp['week'])
tmp = tmp[(tmp['week'].between('2023-03-30', '2025-03-31'))*(tmp['lob']!='Other')].reset_index(drop=True)

# Splitting dataframes by objective
tmp_upper = tmp[tmp['objective']=='Upper Funnel'].reset_index(drop=True)
tmp_cvn = tmp[tmp['objective']=='Conversion'].reset_index(drop=True)

In [0]:
# print(tmp.to_string())

In [0]:
# print(tmp[['lob', 'channel']].drop_duplicates())

#### Flighting

In [0]:
tmp1 = tmp.groupby(['lob', 'channel']).agg({'spend': 'sum', 'revenue': 'sum'}).reset_index() 
tmp1['net_profit'] = tmp1['revenue'] - tmp1['spend']
tmp1['roi'] = tmp1['revenue'] / tmp1['spend'].replace(0, pd.NA)

In [0]:
tmp

In [0]:
# ─── 1. Precompute flighting windows per (lob,channel) ─────────────────────────────
windows = {}

for (chan, lob), grp in tmp.groupby(['channel', 'lob']):
    grp = grp.sort_values('week')
    
    runs = []
    run_start = None
    prev_week = None

    for idx, row in grp.iterrows():
        week = row['week']
        if row['spend'] > 0:
            if run_start is None:
                run_start = week
                prev_week = week
            elif (week - prev_week).days > 8:
                duration = (prev_week + pd.Timedelta(days=7) - run_start).days
                runs.append((date2num(run_start), duration))
                run_start = week
                prev_week = week
            else:
                prev_week = week
        else:
            if run_start is not None:
                duration = (prev_week + pd.Timedelta(days=7) - run_start).days
            runs.append((date2num(run_start), duration))
            run_start = None
        
    # finalize last run
    if run_start is not None:
        duration = (prev_week + pd.Timedelta(days=7) - run_start).days
        runs.append((date2num(run_start), duration))
    windows[(lob,chan)] = runs

# ─── 1.5. Sort key by lob, then channel───────────────────────────────────────────────
sorted_keys = sorted(windows.keys(), key=lambda x: (x[0], x[1]))

# ─── 2. Plot broken horizontal bars ───────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 9), dpi=240)

yticks = []
ylabels = []
height = 0.8
pad = 0.2
for i, key in enumerate(sorted_keys):
    lob, chan = key
    xranges = windows[key]
    y = i * (height + pad)
    color = plt.cm.tab20(i)   # distinct color per series
    ax.broken_barh(
        xranges,
        (y, height),
        facecolors=color,
        edgecolor='k',
        alpha=0.7,
        label=f"{lob} - {chan}"
    )
    yticks.append(y + height/2)
    ylabels.append(f"{lob} / {chan}")

# ─── 3. Formatting ────────────────────────────────────────────────────────────────
ax.set_ylim(-pad, yticks[-1] + height + pad)
ax.set_xlim(date2num(tmp['week'].min()) - 2, date2num(tmp['week'].max()) + 9)
ax.set_yticks(yticks)
ax.set_yticklabels(ylabels)
ax.invert_yaxis()

# show weeks on x‑axis
ax.xaxis_date()
fig.autofmt_xdate()

ax.set_title("Spend Flighting by LOB & Channel")
ax.set_xlabel("Week")
ax.set_ylabel("LOB / Channel")
ax.grid(axis='x', linestyle=':', alpha=0.5)

plt.tight_layout()
plt.show()
        

In [0]:
# display(tmp)

In [0]:
# display(tmp.groupby(['lob', 'channel']).agg({'impressions': 'sum', 'clicks': 'sum'}).reset_index())

In [0]:
tmp_exh = tmp[tmp['lob']=='Exhibition'].groupby('campaign_name').agg({'spend': 'sum'})
tmp_exh.sort_values(by='spend', ascending=False).head(10)

In [0]:
tmp_bra = tmp[tmp['lob']=='Brand'].groupby('campaign_name').agg({'spend': 'sum'})
tmp_bra.sort_values(by='spend', ascending=False).head(10)

#### G&E

In [0]:
# Data Preparation
tmp_gne = tmp[tmp['lob']=='G&E'].groupby('channel').agg({'spend': 'sum', 'clicks': 'sum'})
tmp_gne = tmp_gne / tmp_gne.sum() * 100

# Visualization
indices = np.arange(len(tmp_gne))
fig = plt.figure(figsize=(8, 6), dpi=240)
bars1 = plt.bar(indices - 0.15, tmp_gne['spend'], width=0.3, color=palette[0], zorder=2, label='Spend (% of Total)')
bars2 = plt.bar(indices + 0.15, tmp_gne['clicks'], width=0.3, color=palette[1], zorder=2, label='Clicks (% of Total)')

for bar in bars1 + bars2:
    yval = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2.0, yval + 1, f'{yval:.0f}%', ha='center', va='bottom')

plt.xlabel('Line of Business')
plt.ylabel('% of Total')
plt.xticks(indices, tmp_gne.index)
plt.ylim(0, 100)
plt.grid(axis='y', alpha=0.25)
plt.legend(loc='upper left')
fig.patch.set_facecolor('white')
plt.show()

#### Performance

In [0]:
tmp = pd.read_excel(f'{directory}/raw_data/Datorama Report All Channels Jan 2023 to Apr 2025 Daily Report.xlsx', skiprows=4, skipfooter=1)

tmp.columns = tmp.columns.str.lower().str.replace(' ', '_')
tmp['week'] = pd.to_datetime(tmp['day']).dt.to_period('W-THU').dt.start_time
tmp = tmp[tmp['week'].between('2023-03-31', '2025-03-31')]

In [0]:
tmp_perf = tmp[tmp['line_of_business']=='Performance'].copy()
tmp_perf = tmp_perf.groupby('publisher').agg({'investment': 'sum', 'purchase_revenue': 'sum'})
tmp_perf = tmp_perf / tmp_perf.sum() * 100

# Visualization
indices = np.arange(len(tmp_perf))
fig = plt.figure(figsize=(8, 6), dpi=240)
bars1 = plt.bar(indices - 0.15, tmp_perf['investment'], width=0.3, color=palette[0], zorder=2, label='Spend (% of Total)')
bars2 = plt.bar(indices + 0.15, tmp_perf['purchase_revenue'], width=0.3, color=palette[1], zorder=2, label='Revenue (% of Total)')

for bar in bars1 + bars2:
    yval = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2.0, yval + 1, f'{yval:.0f}%', ha='center', va='bottom')

plt.xlabel('Channel')
plt.ylabel('% of Total')
plt.xticks(indices, tmp_perf.index)
plt.ylim(0, 100)
plt.grid(axis='y', alpha=0.25)
plt.legend(loc='upper left')
fig.patch.set_facecolor('white')
plt.show()

In [0]:
# Data Preparation
tmp_perf1 = tmp[tmp['line_of_business']=='Performance'].copy()
tmp_perf1 = tmp_perf1.groupby(['week', 'publisher']).agg({'investment': 'sum'}).unstack().fillna(0)
tmp_perf1.columns = ['meta_spend', 'programmatic_spend', 'snapchat_spend']
tmp_perf1['total_spend'] = tmp_perf1.sum(axis=1)

tmp_perf2 = tmp[tmp['line_of_business']=='Performance'].copy()
tmp_perf2 = tmp_perf2.groupby('week').agg({'purchase_revenue': 'sum'})

tmp_perf = pd.merge(tmp_perf1, tmp_perf2, left_index=True, right_index=True)

# Visualization
fig, ax1 = plt.subplots(figsize=(9, 7), dpi=240)
ax2 = ax1.twinx()
ax1.bar(tmp_perf.index, tmp_perf['meta_spend'], bottom=tmp_perf['programmatic_spend']+tmp_perf['snapchat_spend'], width=5, color=palette[0], label='Meta', zorder=2)
ax1.bar(tmp_perf.index, tmp_perf['programmatic_spend'], bottom=tmp_perf['snapchat_spend'], width=5, color=palette[0], alpha=0.7, label='Programmatic', zorder=2)
ax1.bar(tmp_perf.index, tmp_perf['snapchat_spend'], width=5, color=palette[0], alpha=0.4, label='Snapchat', zorder=2)
ax2.plot(tmp_perf['purchase_revenue'], color=palette[1], label='Performance Revenue', linewidth=2)
ax1.set_ylim(0, 6*10**4)
ax2.set_ylim(0, 1.2*10**6)
ax1.set_xlabel('Week', fontsize=16)
ax1.set_ylabel('Performance Spend ($)', fontsize=14)
ax2.set_ylabel('Performance Revenue ($)', fontsize=14)
ax1.yaxis.set_major_formatter(mtick.EngFormatter(unit=''))
ax2.yaxis.set_major_formatter(mtick.EngFormatter(unit=''))
ax1.legend(loc='upper left', fontsize=13)
ax2.legend(loc='upper right', fontsize=13)
ax1.tick_params(axis='both', labelsize=12)
ax2.tick_params(axis='both', labelsize=12)
plt.grid(axis='y', alpha=0.25)
fig.patch.set_facecolor('white')
plt.show()

print(tmp_perf[['total_spend', 'purchase_revenue']].corr().iloc[0,1])

In [0]:
# Data Preparation
tmp_perf1 = tmp[tmp['line_of_business']=='Performance'].copy()
tmp_perf1 = tmp_perf1.groupby(['week', 'publisher']).agg({'investment': 'sum'}).unstack().fillna(0)
tmp_perf1.columns = ['meta_spend', 'programmatic_spend', 'snapchat_spend']
tmp_perf1['total_spend'] = tmp_perf1.sum(axis=1)

tmp_perf2 = attendance_and_revenue.groupby('week').agg({'attendance': 'sum'}).copy()

tmp_perf = pd.merge(tmp_perf1, tmp_perf2, left_index=True, right_index=True)

# Visualization
fig, ax1 = plt.subplots(figsize=(9, 7), dpi=240)
ax2 = ax1.twinx()
ax1.bar(tmp_perf.index, tmp_perf['total_spend'], width=5, color=palette[0], label='Performance Spend', zorder=2)
ax2.plot(tmp_perf['attendance'], color=palette[1], label='Attendance', linewidth=2)
ax1.set_ylim(0, 8*10**4)
ax2.set_ylim(0, 2*10**6)
ax1.set_xlabel('Week')
ax1.set_ylabel('Performance Spend ($)')
ax2.set_ylabel('Total Attendance')
ax1.yaxis.set_major_formatter(mtick.EngFormatter(unit=''))
ax2.yaxis.set_major_formatter(mtick.EngFormatter(unit=''))
ax1.legend(loc='upper left')
ax2.legend(loc='upper right')
plt.grid(axis='y', alpha=0.25)
fig.patch.set_facecolor('white')
plt.show()

print(tmp_perf[['total_spend', 'attendance']].corr().iloc[0,1])

#### Prepaid

In [0]:
# Data Preparation
tmp_pre = tmp[tmp['lob']=='Prepaid'].groupby('channel').agg({'spend': 'sum', 'clicks': 'sum'})
tmp_pre = tmp_pre / tmp_pre.sum() * 100

# Visualization
indices = np.arange(len(tmp_pre))
fig = plt.figure(figsize=(8, 6), dpi=240)
bars1 = plt.bar(indices - 0.15, tmp_pre['spend'], width=0.3, color=palette[0], zorder=2, label='Spend (% of Total)')
bars2 = plt.bar(indices + 0.15, tmp_pre['clicks'], width=0.3, color=palette[1], zorder=2, label='Clicks (% of Total)')

for bar in bars1 + bars2:
    yval = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2.0, yval + 1, f'{yval:.0f}%', ha='center', va='bottom')

plt.xlabel('Line of Business')
plt.ylabel('% of Total')
plt.xticks(indices, tmp_pre.index)
plt.ylim(0, 100)
plt.grid(axis='y', alpha=0.25)
plt.legend(loc='upper left')
fig.patch.set_facecolor('white')
plt.show()

#### Upper Funnel LOB Summary

In [0]:
# ─── Data ─────────────────────────────────────────────────────────────────
grouped = tmp_upper.copy()
grouped['net_profit'] = grouped['revenue'] - grouped['spend']
grouped = grouped.groupby('lob').agg({'spend': 'sum', 'revenue': 'sum', 'net_profit': 'sum'}).reset_index()
grouped['roi'] = grouped['revenue'] / grouped['spend']
grouped = grouped.dropna()

# ─── Bubble‐size scaling ──────────────────────────────────────────────────────────
min_size, max_size = 50, 5000
pmin, pmax = grouped['net_profit'].min(), grouped['net_profit'].max()
grouped['size'] = ((grouped['net_profit'] - pmin) / (pmax - pmin)) * (max_size - min_size) + min_size

# ─── Plotting ────────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(11,8), dpi=240)

scatter = ax.scatter(
    grouped['spend'], 
    grouped['roi'], 
    s=grouped['size'], 
    color=palette[0],       
    # alpha=0.6, 
    edgecolors='w'
)

# ─── In‐chart labels ─────────────────────────────────────────────────────────────
for i, row in grouped.iterrows():
    ax.text(row['spend'], row['roi'], f"{row['lob']} (${row['net_profit']:,.0f})", fontsize=7, ha='left', va='top', 
            # backgroundcolor='lightgrey'
            )

# ─── Axis formatting ─────────────────────────────────────────────────────────────
fmt = FuncFormatter(lambda x, pos: f'${x:,.0f}')
ax.xaxis.set_major_formatter(fmt)
ax.yaxis.set_major_formatter(fmt)
ax.margins(x=0.1, y=0.3)
ax.set_xlim(0, 1.2*grouped['spend'].max())
ax.set_ylim(0, 1.2*grouped['roi'].max())

ax.set_xlabel('Spend ($)')
ax.set_ylabel('ROI ($)')
# ax.set_title('Spend vs ROI\n(Bubble size ∝ Net Profit)')
ax.set_title('(Bubble size ∝ Net Profit)')
ax.grid(True, linestyle=':', alpha=0.5)

plt.tight_layout()
plt.show()

#### Lower Funnel LOB Summary

In [0]:
# ─── Data ─────────────────────────────────────────────────────────────────
grouped = tmp_cvn.copy()
grouped['net_profit'] = grouped['revenue'] - grouped['spend']
grouped = grouped.groupby('lob').agg({'spend': 'sum', 'revenue': 'sum', 'net_profit': 'sum'}).reset_index()
grouped['roi'] = grouped['revenue'] / grouped['spend']

# ─── Bubble‐size scaling ──────────────────────────────────────────────────────────
min_size, max_size = 50, 5000
pmin, pmax = grouped['net_profit'].min(), grouped['net_profit'].max()
grouped['size'] = ((grouped['net_profit'] - pmin) / (pmax - pmin)) * (max_size - min_size) + min_size

# ─── Plotting ────────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(11,8), dpi=240)

scatter = ax.scatter(
    grouped['spend'], 
    grouped['roi'], 
    s=grouped['size'], 
    color=palette[0],       
    # alpha=0.6, 
    edgecolors='w'
)

# ─── In‐chart labels ─────────────────────────────────────────────────────────────
for i, row in grouped.iterrows():
    ax.text(row['spend'], row['roi'], f"{row['lob']} (${row['net_profit']:,.0f})", fontsize=7, ha='left', va='top', 
            # backgroundcolor='lightgrey'
            )

# ─── Axis formatting ─────────────────────────────────────────────────────────────
fmt = FuncFormatter(lambda x, pos: f'${x:,.0f}')
ax.xaxis.set_major_formatter(fmt)
ax.yaxis.set_major_formatter(fmt)
ax.margins(x=0.1, y=0.3)
ax.set_xlim(0, 1.2*grouped['spend'].max())
ax.set_ylim(0, 1.2*grouped['roi'].max())

ax.set_xlabel('Spend ($)')
ax.set_ylabel('ROI ($)')
# ax.set_title('Spend vs ROI\n(Bubble size ∝ Net Profit)')
ax.set_title('(Bubble size ∝ Net Profit)')
ax.grid(True, linestyle=':', alpha=0.5)

plt.tight_layout()
plt.show()

#### Upper Funnel Channel Summary

In [0]:
# ─── Data ─────────────────────────────────────────────────────────────────
grouped = tmp_upper.copy()
grouped['net_profit'] = grouped['revenue'] - grouped['spend']
grouped = grouped.groupby('channel').agg({'spend': 'sum', 'revenue': 'sum', 'net_profit': 'sum'}).reset_index()
grouped['roi'] = grouped['revenue'] / grouped['spend']

# ─── Bubble‐size scaling ──────────────────────────────────────────────────────────
min_size, max_size = 50, 5000
pmin, pmax = grouped['net_profit'].min(), grouped['net_profit'].max()
grouped['size'] = ((grouped['net_profit'] - pmin) / (pmax - pmin)) * (max_size - min_size) + min_size

# ─── Plotting ────────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(11,8), dpi=240)

scatter = ax.scatter(
    grouped['spend'], 
    grouped['roi'], 
    s=grouped['size'], 
    color=palette[0],       
    # alpha=0.6, 
    edgecolors='w'
)

# ─── In‐chart labels ─────────────────────────────────────────────────────────────
for i, row in grouped.iterrows():
    ax.text(row['spend'], row['roi'], f"{row['channel']} (${row['net_profit']:,.0f})", fontsize=7, ha='left', va='top', 
            # backgroundcolor='lightgrey'
            )

# ─── Axis formatting ─────────────────────────────────────────────────────────────
fmt = FuncFormatter(lambda x, pos: f'${x:,.0f}')
ax.xaxis.set_major_formatter(fmt)
ax.yaxis.set_major_formatter(fmt)
ax.margins(x=0.1, y=0.3)
ax.set_xlim(0, 1.2*grouped['spend'].max())
ax.set_ylim(0, 1.2*grouped['roi'].max())

ax.set_xlabel('Spend ($)')
ax.set_ylabel('ROI ($)')
# ax.set_title('Spend vs ROI\n(Bubble size ∝ Net Profit)')
ax.set_title('(Bubble size ∝ Net Profit)')
ax.grid(True, linestyle=':', alpha=0.5)

plt.tight_layout()
plt.show()

#### Lower Funnel Channel Summary

In [0]:
# ─── Data ─────────────────────────────────────────────────────────────────
grouped = tmp_cvn.copy()
grouped['net_profit'] = grouped['revenue'] - grouped['spend']
grouped = grouped.groupby('channel').agg({'spend': 'sum', 'revenue': 'sum', 'net_profit': 'sum'}).reset_index()
grouped['roi'] = grouped['revenue'] / grouped['spend']

# ─── Bubble‐size scaling ──────────────────────────────────────────────────────────
min_size, max_size = 50, 5000
pmin, pmax = grouped['net_profit'].min(), grouped['net_profit'].max()
grouped['size'] = ((grouped['net_profit'] - pmin) / (pmax - pmin)) * (max_size - min_size) + min_size

# ─── Plotting ────────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(11,8), dpi=240)

scatter = ax.scatter(
    grouped['spend'], 
    grouped['roi'], 
    s=grouped['size'], 
    color=palette[0],       
    # alpha=0.6, 
    edgecolors='w'
)

# ─── In‐chart labels ─────────────────────────────────────────────────────────────
for i, row in grouped.iterrows():
    ax.text(row['spend'], row['roi'], f"{row['channel']} (${row['net_profit']:,.0f})", fontsize=7, ha='left', va='top', 
            # backgroundcolor='lightgrey'
            )

# ─── Axis formatting ─────────────────────────────────────────────────────────────
fmt = FuncFormatter(lambda x, pos: f'${x:,.0f}')
ax.xaxis.set_major_formatter(fmt)
ax.yaxis.set_major_formatter(fmt)
ax.margins(x=0.1, y=0.3)
ax.set_xlim(0, 1.2*grouped['spend'].max())
ax.set_ylim(0, 1.2*grouped['roi'].max())

ax.set_xlabel('Spend ($)')
ax.set_ylabel('ROI ($)')
# ax.set_title('Spend vs ROI\n(Bubble size ∝ Net Profit)')
ax.set_title('(Bubble size ∝ Net Profit)')
ax.grid(True, linestyle=':', alpha=0.5)

plt.tight_layout()
plt.show()

#### Cineclub Channel Summary

In [0]:
# ─── Data ─────────────────────────────────────────────────────────────────
grouped = tmp[tmp['lob']=='Cineclub'].copy()
grouped['net_profit'] = grouped['revenue'] - grouped['spend']
grouped = grouped.groupby('channel').agg({'spend': 'sum', 'revenue': 'sum', 'net_profit': 'sum'}).reset_index()
grouped['roi'] = grouped['revenue'] / grouped['spend']

# ─── Bubble‐size scaling ──────────────────────────────────────────────────────────
min_size, max_size = 50, 5000
pmin, pmax = grouped['net_profit'].min(), grouped['net_profit'].max()
grouped['size'] = ((grouped['net_profit'] - pmin) / (pmax - pmin)) * (max_size - min_size) + min_size

# ─── Plotting ────────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(11,8), dpi=240)

scatter = ax.scatter(
    grouped['spend'], 
    grouped['roi'], 
    s=grouped['size'], 
    color=palette[0],       
    # alpha=0.6, 
    edgecolors='w'
)

# ─── In‐chart labels ─────────────────────────────────────────────────────────────
for i, row in grouped.iterrows():
    ax.text(row['spend'], row['roi'], f"{row['channel']} (${row['net_profit']:,.0f})", fontsize=7, ha='left', va='top', 
            # backgroundcolor='lightgrey'
            )

# ─── Axis formatting ─────────────────────────────────────────────────────────────
fmt = FuncFormatter(lambda x, pos: f'${x:,.0f}')
ax.xaxis.set_major_formatter(fmt)
ax.yaxis.set_major_formatter(fmt)
ax.margins(x=0.1, y=0.3)
ax.set_xlim(0, 1.2*grouped['spend'].max())
ax.set_ylim(0, 1.2*grouped['roi'].max())

ax.set_xlabel('Spend ($)')
ax.set_ylabel('ROI ($)')
# ax.set_title('Spend vs ROI\n(Bubble size ∝ Net Profit)')
ax.set_title('(Bubble size ∝ Net Profit)')
ax.grid(True, linestyle=':', alpha=0.5)

plt.tight_layout()
plt.show()

#### IMAX Channel Summary

In [0]:
# ─── Data ─────────────────────────────────────────────────────────────────
grouped = tmp[tmp['lob'].isin(['IMAX', 'Enhanced Experiences'])].copy()
grouped['net_profit'] = grouped['revenue'] - grouped['spend']
grouped = grouped.groupby('channel').agg({'spend': 'sum', 'revenue': 'sum', 'net_profit': 'sum'}).reset_index()
grouped['roi'] = grouped['revenue'] / grouped['spend']

# ─── Bubble‐size scaling ──────────────────────────────────────────────────────────
min_size, max_size = 50, 5000
pmin, pmax = grouped['net_profit'].min(), grouped['net_profit'].max()
grouped['size'] = ((grouped['net_profit'] - pmin) / (pmax - pmin)) * (max_size - min_size) + min_size

# ─── Plotting ────────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(11,8), dpi=240)

scatter = ax.scatter(
    grouped['spend'], 
    grouped['roi'], 
    s=grouped['size'], 
    color=palette[0],       
    # alpha=0.6, 
    edgecolors='w'
)

# ─── In‐chart labels ─────────────────────────────────────────────────────────────
for i, row in grouped.iterrows():
    ax.text(row['spend'], row['roi'], f"{row['channel']} (${row['net_profit']:,.0f})", fontsize=7, ha='left', va='top', 
            # backgroundcolor='lightgrey'
            )

# ─── Axis formatting ─────────────────────────────────────────────────────────────
fmt = FuncFormatter(lambda x, pos: f'${x:,.0f}')
ax.xaxis.set_major_formatter(fmt)
ax.yaxis.set_major_formatter(fmt)
ax.margins(x=0.1, y=0.3)
ax.set_xlim(0, 1.2*grouped['spend'].max())
ax.set_ylim(0, 1.2*grouped['roi'].max())

ax.set_xlabel('Spend ($)')
ax.set_ylabel('ROI ($)')
# ax.set_title('Spend vs ROI\n(Bubble size ∝ Net Profit)')
ax.set_title('(Bubble size ∝ Net Profit)')
ax.grid(True, linestyle=':', alpha=0.5)

plt.tight_layout()
plt.show()